# Chapter 21 — Panel Data across Indian States

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch21_panel_data_states.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 21.1 Per-capita income by state

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

from indiadata import mospi_nas

pc = mospi_nas(25)                      # per capita NSDP, rupees
print(pc["indicator"].unique()[0], "| unit:", pc["unit"].unique()[0])
print(pc.shape, "rows;", pc["state"].nunique(), "states/UTs;",
      pc["year"].min(), "to", pc["year"].max())
pc[["state", "year", "fy", "current_price", "constant_price"]].head(3)

In [ ]:
pc = pc.sort_values(["state", "fy"])
pc["my_growth"] = 100 * pc.groupby("state")["constant_price"].pct_change()

published = mospi_nas(28)[["state", "fy", "constant_price"]].rename(
    columns={"constant_price": "MoSPI_growth"})
check = pc.merge(published, on=["state", "fy"])
gap = (check["my_growth"].round(1) - check["MoSPI_growth"]).abs()
print(f"{len(check)} state-years compared; largest gap {gap.max():.1f} "
      f"percentage points")
check.loc[check["state"] == "Bihar", ["year", "my_growth", "MoSPI_growth"]
          ].head(4).round(1)

In [ ]:
nsdp = mospi_nas("NSDP")[["state", "fy", "current_price"]]
pop = nsdp.merge(pc[["state", "fy", "current_price"]], on=["state", "fy"],
                 suffixes=("_nsdp", "_pc"))
pop["pop_crore"] = pop["current_price_nsdp"] * 1e7 / pop["current_price_pc"] / 1e7
implied = pop.pivot(index="state", columns="fy", values="pop_crore")
implied[[2011, 2024]].sort_values(2011, ascending=False).head(5).round(2)

In [ ]:
alt = mospi_nas(32).query("state == 'Karnataka'").set_index("fy")
alt_gsdp = mospi_nas(31).query("state == 'Karnataka'").set_index("fy")
ours = pc.query("state == 'Karnataka'").set_index("fy")
pd.DataFrame({"code 25 constant": ours["constant_price"],
              "code 32 constant": alt["constant_price"],
              "code 31 current": alt_gsdp["current_price"]}).loc[[2011, 2016, 2024]]

## 21.2 Long and wide; balanced and unbalanced

In [ ]:
wide = pc.pivot(index="state", columns="fy", values="constant_price")
print("wide:", wide.shape, "  long:", pc.shape)
wide.iloc[:4, -4:]

In [ ]:
wide.notna().sum().tail(3)

In [ ]:
d = pc.loc[pc["fy"] <= 2024, ["state", "fy", "year", "constant_price"]].copy()
d = d.rename(columns={"constant_price": "pcnsdp"})
d["ly"] = np.log(d["pcnsdp"])                                   # log income
d["growth"] = 100 * d.groupby("state")["ly"].diff()             # % per year
d["ly_lag"] = d.groupby("state")["ly"].shift()                  # last year's log income
print("balanced:", d.groupby("state").size().nunique() == 1,
      "| states:", d["state"].nunique(), "| years:", d["fy"].nunique())

## 21.3 β-convergence in a cross-section

In [ ]:
cs = pd.DataFrame({
    "growth": 100 * (wide[2024].pipe(np.log) - wide[2011].pipe(np.log)) / 13,
    "ly0": np.log(wide[2011]),
})
cs["major"] = implied[2011] > 2           # population above 2 crore in 2011-12

ols_all = smf.ols("growth ~ ly0", data=cs).fit(cov_type="HC1")
ols_major = smf.ols("growth ~ ly0", data=cs[cs["major"]]).fit(cov_type="HC1")
for name, r in [("All 30 states/UTs", ols_all), ("16 major states", ols_major)]:
    print(f"{name:18s} b = {r.params['ly0']:6.2f}  (robust SE {r.bse['ly0']:.2f}, "
          f"p = {r.pvalues['ly0']:.3f}),  R² = {r.rsquared:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for major, colour, label in [(False, "#9db7d5", "Smaller states and UTs"),
                             (True, "#1f4e79", "16 major states")]:
    sub = cs[cs["major"] == major]
    ax.scatter(sub["ly0"], sub["growth"], color=colour, s=45, label=label,
               zorder=3)
x = np.linspace(cs["ly0"].min(), cs["ly0"].max(), 50)
ax.plot(x, ols_all.params["Intercept"] + ols_all.params["ly0"] * x,
        color="#9db7d5", ls="--", lw=1.2)
xm = np.linspace(cs.loc[cs["major"], "ly0"].min(), cs.loc[cs["major"], "ly0"].max(), 50)
ax.plot(xm, ols_major.params["Intercept"] + ols_major.params["ly0"] * xm,
        color="#1f4e79", lw=1.5)
offsets = {"Telangana": (-8, -12), "Kerala": (6, -6)}   # move labels off other dots
for s in ["Bihar", "Uttar Pradesh", "Gujarat", "Karnataka", "Kerala",
          "Maharashtra", "West Bengal", "Telangana", "Goa", "Delhi", "Mizoram",
          "Punjab"]:
    ax.annotate(s, (cs.loc[s, "ly0"], cs.loc[s, "growth"]), fontsize=8,
                xytext=offsets.get(s, (4, 3)), textcoords="offset points")
ax.set_xlabel("Log real per capita NSDP, 2011–12")
ax.set_ylabel("Average growth, 2011–12 to 2024–25, % a year")
ax.set_title("Did poorer states grow faster?", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Lines: OLS fits (dashed: all 30; solid: major states). "
         "Source: MoSPI NAS, per capita NSDP at 2011–12 prices (API code 25).",
         fontsize=8, color="grey")
plt.show()

In [ ]:
def speed(b, T=13):
    """Annual convergence speed implied by a cross-section slope b (in %)."""
    lam = -np.log(1 + b * T / 100) / T
    return lam, np.log(2) / lam

lam, half = speed(ols_all.params["ly0"])
print(f"All states: lambda = {lam:.2%} a year, half-life {half:.0f} years")

## 21.4 From a cross-section to a panel

In [ ]:
from linearmodels.panel import PanelOLS, PooledOLS, BetweenOLS, RandomEffects

p = d.dropna(subset=["growth"]).set_index(["state", "fy"])
print(p.shape, "observations: 30 states × 13 growth years")
y, X = p["growth"], sm.add_constant(p[["ly_lag"]])

In [ ]:
pooled = PooledOLS(y, X).fit(cov_type="clustered", cluster_entity=True)
between = BetweenOLS(y, X).fit()
for name, r in [("Pooled OLS", pooled), ("Between", between)]:
    print(f"{name:11s} beta = {r.params['ly_lag']:6.2f}  "
          f"(SE {r.std_errors['ly_lag']:.2f})")

## 21.5 Fixed effects: comparing each state with itself

In [ ]:
demeaned = p[["growth", "ly_lag"]] - p.groupby("state")[["growth", "ly_lag"]].transform("mean")
by_hand = sm.OLS(demeaned["growth"], demeaned["ly_lag"]).fit()
print(f"FE by hand: beta = {by_hand.params['ly_lag']:.3f}")
print("Check: every state's demeaned growth averages to",
      f"{demeaned.groupby('state')['growth'].mean().abs().max():.1e}")

In [ ]:
fe = PanelOLS(y, p[["ly_lag"]], entity_effects=True).fit()
print(f"PanelOLS FE: beta = {fe.params['ly_lag']:.3f}  (SE {fe.std_errors['ly_lag']:.3f})")
print(f"By-hand SE:  {by_hand.bse['ly_lag']:.3f}")

In [ ]:
lsdv = smf.ols("growth ~ ly_lag + C(state)", data=p.reset_index()).fit()
print(f"LSDV: beta = {lsdv.params['ly_lag']:.3f}  (SE {lsdv.bse['ly_lag']:.3f}), "
      f"{len(lsdv.params)} coefficients")

## 21.6 Random effects and the Hausman test

In [ ]:
re = RandomEffects(y, X).fit()
print(f"RE: beta = {re.params['ly_lag']:.3f} (SE {re.std_errors['ly_lag']:.3f}),"
      f"  theta = {re.theta.iloc[0, 0]:.2f}")
print(f"SD across states of average growth: {p.groupby('state')['growth'].mean().std():.2f}; "
      f"SD of year-to-year deviations: {demeaned['growth'].std():.2f}; "
      f"noise alone would give {demeaned['growth'].std() / np.sqrt(13):.2f}")
re.variance_decomposition

In [ ]:
def hausman(fe_res, re_res, cols):
    """Hausman test of FE against RE for the coefficients in `cols`."""
    b = fe_res.params[cols] - re_res.params[cols]
    V = fe_res.cov.loc[cols, cols] - re_res.cov.loc[cols, cols]
    H = float(b @ np.linalg.inv(V) @ b)
    return H, stats.chi2.sf(H, df=len(cols))

H, pval = hausman(fe, re, ["ly_lag"])
print(f"Hausman statistic = {H:.2f}, df = 1, p-value = {pval:.4f}")

## 21.7 Clustered standard errors and two-way fixed effects

In [ ]:
se = pd.DataFrame({
    cov: PanelOLS(y, p[["ly_lag"]], entity_effects=True)
         .fit(**kw).std_errors["ly_lag"]
    for cov, kw in [("classical", {}),
                    ("robust (White)", {"cov_type": "robust"}),
                    ("clustered by state", {"cov_type": "clustered",
                                            "cluster_entity": True})]
}, index=["SE of beta"]).round(3)
se

In [ ]:
fe2 = PanelOLS(y, p[["ly_lag"]], entity_effects=True, time_effects=True).fit(
    cov_type="clustered", cluster_entity=True)
print(f"Two-way FE: beta = {fe2.params['ly_lag']:.2f}  "
      f"(clustered SE {fe2.std_errors['ly_lag']:.2f})")
cross_avg = p.groupby(level="fy")["growth"].mean()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
labels = [f"{t}-{str(t + 1)[2:]}" for t in cross_avg.index]
ax.bar(labels, cross_avg, color=["#c55a11" if t == 2020 else "#9db7d5"
                                 for t in cross_avg.index])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Average real growth across 30 states, %")
ax.set_title("The year effect: one shock hit every state at once", loc="left",
             fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.text(0.01, -0.1, "Unweighted mean of per capita NSDP growth (log change × 100). "
         "Source: MoSPI NAS (API code 25).", fontsize=8, color="grey")
plt.show()
print(f"Average growth 2020-21: {cross_avg[2020]:.1f}%;  "
      f"2021-22: {cross_avg[2021]:.1f}%")

## 21.8 The Nickell bias: why FE misleads in a short dynamic panel

In [ ]:
rng = np.random.default_rng(2021)
mu = p.groupby("state")["growth"].mean().to_numpy()       # each state's own trend
sd = demeaned["growth"].std()                              # size of yearly shocks
ly0 = np.log(wide[2011]).to_numpy()
R, N, T = 1000, len(mu), 13

shocks = rng.normal(0, sd, size=(R, N, T))
g_sim = mu[None, :, None] + shocks                         # growth, % a year
ly_sim = ly0[None, :, None] + np.cumsum(g_sim, axis=2) / 100
lag_sim = np.concatenate([np.broadcast_to(ly0[None, :, None], (R, N, 1)),
                          ly_sim[:, :, :-1]], axis=2)       # ln y_{t-1}

gd = g_sim - g_sim.mean(axis=2, keepdims=True)              # within transformation
xd = lag_sim - lag_sim.mean(axis=2, keepdims=True)
beta_sim = (gd * xd).sum(axis=(1, 2)) / (xd ** 2).sum(axis=(1, 2))
print(f"True beta = 0.  Mean FE estimate = {beta_sim.mean():.1f}; "
      f"share of estimates below our {fe.params['ly_lag']:.1f}: "
      f"{(beta_sim < fe.params['ly_lag']).mean():.0%}")
del shocks, g_sim, ly_sim, lag_sim, gd, xd

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(beta_sim, bins=40, color="#9db7d5", edgecolor="white")
ax.axvline(0, color="black", lw=1, ls="--")
ax.axvline(fe.params["ly_lag"], color="#c55a11", lw=2)
ax.text(fe.params["ly_lag"], ax.get_ylim()[1] * 0.92,
        f"our FE estimate ({fe.params['ly_lag']:.1f})  ", color="#c55a11",
        fontsize=9, ha="right")
ax.text(0, ax.get_ylim()[1] * 0.80, "  true value (0)", fontsize=9)
ax.set_xlabel("FE estimate of beta in a simulated panel with no convergence")
ax.set_ylabel("Number of simulations")
ax.set_title("Simulation: FE finds 'convergence' even when there is none",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.05, "1,000 simulated panels, 30 states × 13 years; trends and "
         "shock size calibrated to MoSPI per capita NSDP. Simulated, not data.",
         fontsize=8, color="grey")
plt.show()

## 21.9 Summary table

In [ ]:
models = {"Cross-section": (ols_all.params["ly0"], ols_all.bse["ly0"], 30),
          "Pooled OLS": (pooled.params["ly_lag"], pooled.std_errors["ly_lag"], pooled.nobs),
          "Between": (between.params["ly_lag"], between.std_errors["ly_lag"], between.nobs),
          "Random effects": (re.params["ly_lag"], re.std_errors["ly_lag"], re.nobs)}
fe_cl = PanelOLS(y, p[["ly_lag"]], entity_effects=True).fit(
    cov_type="clustered", cluster_entity=True)
models["FE (state)"] = (fe_cl.params["ly_lag"], fe_cl.std_errors["ly_lag"], fe_cl.nobs)
models["FE (state + year)"] = (fe2.params["ly_lag"], fe2.std_errors["ly_lag"], fe2.nobs)
table = pd.DataFrame(models, index=["beta", "SE", "obs"]).T
table["obs"] = table["obs"].astype(int)
table.round(2)

## 21.10 σ-convergence: is the spread narrowing?

In [ ]:
sigma = pd.DataFrame({
    "All 30": np.log(wide.loc[:, 2011:2024]).std(),
    "16 major states": np.log(wide.loc[cs["major"], 2011:2024]).std(),
})
sigma.iloc[[0, 4, 9, 13]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(sigma.index, sigma["All 30"], marker="o", color="#9db7d5",
        label="All 30 states and UTs")
ax.plot(sigma.index, sigma["16 major states"], marker="o", color="#1f4e79",
        label="16 major states")
ax.set_xticks(sigma.index)
ax.set_xticklabels([f"{t}-{str(t + 1)[2:]}" for t in sigma.index], rotation=45,
                   ha="right")
ax.set_ylabel("SD of log per capita NSDP")
ax.set_title("σ-convergence: the spread between states", loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.11, "Real per capita NSDP at 2011–12 prices, unweighted. "
         "Source: MoSPI NAS (API code 25).", fontsize=8, color="grey")
plt.show()

In [ ]:
for col in sigma:
    print(f"{col:16s}: {sigma[col].iloc[0]:.3f} in 2011-12 -> "
          f"{sigma[col].iloc[-1]:.3f} in 2024-25 "
          f"(peak {sigma[col].max():.3f} in {sigma[col].idxmax()}-{str(sigma[col].idxmax() + 1)[2:]})")

## 21.11 What can a 14-year panel tell us?

In [ ]:
print(f"Population, 2024-25: Uttar Pradesh {implied.loc['Uttar Pradesh', 2024]:.2f} crore, "
      f"Sikkim {implied.loc['Sikkim', 2024]:.3f} crore; ratio "
      f"{implied.loc['Uttar Pradesh', 2024] / implied.loc['Sikkim', 2024]:.0f}")

## Key ideas

## Exercises

## Data citation

## Further reading